In [3]:
# ODIN EXECUTIVE SUITE v8.0 (CEO COMMAND CENTER)
# "The War Room" - Visualizing Survival, Cash, and Strategy.

import os
import io
import xmlrpc.client
import pandas as pd
import numpy as np
import json
import matplotlib.pyplot as plt
from typing import Dict, Any, Optional, List
from datetime import datetime, timedelta
from dataclasses import dataclass

# Environment
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI

# PDF Generation (ReportLab)
from reportlab.lib.pagesizes import letter
from reportlab.platypus import (
    SimpleDocTemplate, Paragraph, Spacer, Table, 
    TableStyle, PageBreak, Image as RLImage
)
from reportlab.lib.styles import getSampleStyleSheet, ParagraphStyle
from reportlab.lib import colors
from reportlab.lib.units import inch

# ================================================================
# 1. CONFIGURATION
# ================================================================
load_dotenv()
# Force matplotlib to not use Xserver (safe for servers)
plt.switch_backend('Agg')

class Config:
    # Odoo Credentials
    ODOO_URL = os.getenv('ODOO_URL', 'https://vendyltd.odoo.com/')
    ODOO_DB = os.getenv('ODOO_DB', 'vendyltd')
    ODOO_USER = os.getenv('ODOO_USER', 'muktadir@vendy.ltd')
    ODOO_PASSWORD = os.getenv('ODOO_PASSWORD', '205958c9bb841b4bef7e6da4a3afb5b5029cd6ae')
    
    # AI Settings
    LLM_MODEL = "gpt-4-turbo"
    REPORT_DIR = "./odin_reports"
    
    @classmethod
    def ensure_dirs(cls):
        os.makedirs(cls.REPORT_DIR, exist_ok=True)

Config.ensure_dirs()

# ================================================================
# 2. DATA MODELS & UTILS
# ================================================================
@dataclass
class AgentResult:
    data: Dict[str, Any]
    timestamp: datetime
    success: bool
    error: Optional[str] = None

class SafetyUtils:
    @staticmethod
    def safe_df(records: List[Dict], required_cols: List[str]) -> pd.DataFrame:
        if not records: return pd.DataFrame(columns=required_cols)
        try:
            df = pd.DataFrame(records)
            for col in required_cols:
                if col not in df.columns: df[col] = 0
            return df
        except Exception:
            return pd.DataFrame(columns=required_cols)

# ================================================================
# 3. ODOO CONNECTION
# ================================================================
class OdooClient:
    def __init__(self):
        self.connect()
    
    def connect(self):
        try:
            common = xmlrpc.client.ServerProxy(f"{Config.ODOO_URL}/xmlrpc/2/common")
            self.uid = common.authenticate(Config.ODOO_DB, Config.ODOO_USER, Config.ODOO_PASSWORD, {})
            self.models = xmlrpc.client.ServerProxy(f"{Config.ODOO_URL}/xmlrpc/2/object")
            self.connected = bool(self.uid)
            print("✅ Connected to Odoo" if self.connected else "❌ Authentication failed")
        except Exception as e:
            print(f"❌ Odoo connection failed: {e}")
            self.connected = False
    
    def read(self, model: str, domain: List, fields: List) -> List[Dict]:
        if not self.connected: return []
        return self.models.execute_kw(
            Config.ODOO_DB, self.uid, Config.ODOO_PASSWORD,
            model, 'search_read', [domain], {'fields': fields}
        )

odoo = OdooClient()

# ================================================================
# 4. VISUALIZATION ENGINE (The "CEO Eyes")
# ================================================================
class VisualizerEngine:
    """Generates charts that answer 'Are we dying?' and 'Who owes us?'"""
    
    @staticmethod
    def _save_plot():
        buf = io.BytesIO()
        plt.savefig(buf, format='png', dpi=150, bbox_inches='tight')
        plt.close()
        buf.seek(0)
        return buf

    @staticmethod
    def plot_profit_paradox(profit, cash_flow):
        """Bar chart comparing Paper Profit vs Real Cash"""
        labels = ['Paper Profit', 'Real Cash Flow']
        values = [profit, cash_flow]
        colors_list = ['#2ecc71', '#e74c3c'] if cash_flow < profit else ['#2ecc71', '#27ae60']
        
        plt.figure(figsize=(5, 3))
        bars = plt.bar(labels, values, color=colors_list, width=0.6)
        
        # Labels
        for bar in bars:
            height = bar.get_height()
            plt.text(bar.get_x() + bar.get_width()/2., height,
                     f'${height:,.0f}', ha='center', va='bottom', fontsize=9, fontweight='bold')

        plt.title('THE PROFIT PARADOX (Income vs Cash)', fontsize=10, fontweight='bold')
        plt.axhline(0, color='black', linewidth=0.8)
        plt.grid(axis='y', linestyle='--', alpha=0.3)
        return VisualizerEngine._save_plot()

    @staticmethod
    def plot_cash_runway(current_cash, burn_rate, months=6):
        """Line chart projecting when money runs out"""
        months_arr = range(months + 1)
        # Prevent division by zero if burn_rate is 0 or negative (unlikely for expense)
        effective_burn = burn_rate if burn_rate > 0 else 1000 
        cash_balance = [current_cash - (effective_burn * m) for m in months_arr]
        
        plt.figure(figsize=(5, 3))
        plt.plot(months_arr, cash_balance, marker='o', color='#c0392b', linewidth=2)
        plt.axhline(0, color='black', linestyle='--')
        
        # Shade the "Death Zone" (Negative Cash)
        plt.fill_between(months_arr, cash_balance, 0, where=[c < 0 for c in cash_balance], color='red', alpha=0.2, interpolate=True)
        
        plt.title(f'SURVIVAL FORECAST (Burn: ${burn_rate:,.0f}/mo)', fontsize=10, fontweight='bold')
        plt.xlabel('Months from Now')
        plt.ylabel('Projected Cash ($)')
        plt.grid(True, alpha=0.3)
        return VisualizerEngine._save_plot()

    @staticmethod
    def plot_debtor_breakdown(debtor_data):
        """Horizontal bar chart of Top 5 debtors"""
        if not debtor_data: return None
        
        # Top 5 by amount
        sorted_debtors = dict(sorted(debtor_data.items(), key=lambda item: item[1], reverse=True)[:5])
        
        names = list(sorted_debtors.keys())
        amounts = list(sorted_debtors.values())
        
        plt.figure(figsize=(5, 3))
        bars = plt.barh(names, amounts, color='#e67e22')
        
        # Add values
        for index, value in enumerate(amounts):
            plt.text(value, index, f' ${value:,.0f}', va='center', fontsize=8)

        plt.xlabel('Amount ($)')
        plt.title('TOP 5 DEBTORS (CALL LIST)', fontsize=10, fontweight='bold')
        plt.tight_layout()
        return VisualizerEngine._save_plot()

    @staticmethod
    def plot_inventory_risk(good, ghost):
        """Pie chart of Ghost Assets"""
        labels = ['Healthy Stock', 'Ghost Assets (Neg)']
        sizes = [good, ghost]
        colors_list = ['#3498db', '#c0392b']
        explode = (0, 0.1)

        plt.figure(figsize=(4, 3))
        plt.pie(sizes, explode=explode, labels=labels, colors=colors_list, autopct='%1.1f%%', shadow=True, startangle=90)
        plt.title('INVENTORY INTEGRITY', fontsize=10, fontweight='bold')
        return VisualizerEngine._save_plot()

# ================================================================
# 5. AGENTS (Auditor, Financialist, Strategist)
# ================================================================

class AuditorAgent:
    """Finds 'Phantom Revenue' and 'Ghost Assets'"""
    def execute(self, d_from, d_to) -> AgentResult:
        try:
            # 1. Revenue Integrity (Invoice vs Sales Gap)
            sales = odoo.read('sale.order', [('date_order', '>=', d_from), ('date_order', '<=', d_to), ('state', 'in', ['sale', 'done'])], ['amount_total'])
            invoices = odoo.read('account.move', [('date', '>=', d_from), ('date', '<=', d_to), ('move_type', '=', 'out_invoice'), ('state', '=', 'posted')], ['amount_total'])
            
            total_sales = sum(x['amount_total'] for x in sales)
            total_invoiced = sum(x['amount_total'] for x in invoices)
            gap = total_invoiced - total_sales
            
            # 2. Inventory Integrity (Negative Stock)
            quants = odoo.read('stock.quant', [], ['quantity'])
            ghost_stock = len([x for x in quants if x['quantity'] < 0])
            good_stock = len([x for x in quants if x['quantity'] > 0])
            
            findings = []
            if abs(gap) > 1000: findings.append(f"REVENUE GAP: Invoiced ${total_invoiced:,.0f} vs Sold ${total_sales:,.0f}. Variance: ${gap:,.0f} (Potential Fraud/Error).")
            if ghost_stock > 0: findings.append(f"GHOST ASSETS: {ghost_stock} products show negative stock. COGS is unreliable.")

            return AgentResult({
                "revenue_gap": gap,
                "ghost_stock": ghost_stock,
                "good_stock": good_stock,
                "findings": findings
            }, datetime.now(), True)
        except Exception as e:
            return AgentResult({}, datetime.now(), False, str(e))

class FinancialistAgent:
    """Calculates Survival Metrics & Drill-Downs"""
    def execute(self, d_from, d_to) -> AgentResult:
        try:
            # Fetch Financial Moves
            domain = [('state', '=', 'posted'), ('date', '>=', d_from), ('date', '<=', d_to)]
            moves = odoo.read('account.move', domain, ['amount_total', 'move_type', 'payment_state', 'partner_id', 'amount_residual'])
            df = SafetyUtils.safe_df(moves, ['amount_total', 'move_type', 'payment_state', 'amount_residual'])
            
            # P&L
            revenue = df[df['move_type'] == 'out_invoice']['amount_total'].sum()
            expenses = df[df['move_type'] == 'in_invoice']['amount_total'].sum()
            profit = revenue - expenses
            
            # Cash Flow (Proxy: Paid In - Paid Out)
            cash_in = df[(df['move_type'] == 'out_invoice') & (df['payment_state'] == 'paid')]['amount_total'].sum()
            cash_out = df[(df['move_type'] == 'in_invoice') & (df['payment_state'] == 'paid')]['amount_total'].sum()
            net_cash = cash_in - cash_out
            
            # Burn Rate (Monthly Average Expenses)
            burn_rate = expenses / 3  # Assuming 90 day period
            
            # Debtor Drill-Down (Who owes us?)
            # Filter for unpaid invoices
            unpaid_invs = [m for m in moves if m['move_type'] == 'out_invoice' and m['payment_state'] != 'paid']
            debtors = {}
            for inv in unpaid_invs:
                if inv.get('partner_id'):
                    name = inv['partner_id'][1] # [id, "Name"]
                    amt = inv.get('amount_residual', 0)
                    debtors[name] = debtors.get(name, 0) + amt

            return AgentResult({
                "profit": profit,
                "net_cash_flow": net_cash,
                "burn_rate": burn_rate,
                "debtors": debtors,
                "total_ar": sum(debtors.values()),
                "revenue": revenue,
                "expenses": expenses
            }, datetime.now(), True)
        except Exception as e:
            return AgentResult({}, datetime.now(), False, str(e))

class StrategistAgent:
    """Generates Orders via LLM"""
    def __init__(self):
        self.llm = ChatOpenAI(model=Config.LLM_MODEL, temperature=0.3)
    
    def execute(self, audit, fin) -> AgentResult:
        try:
            prompt = f"""You are a Turnaround CEO. Review this critical data:
            - CASH FLOW: ${fin.get('net_cash_flow', 0):,.0f}
            - PROFIT: ${fin.get('profit', 0):,.0f}
            - UNCOLLECTED AR: ${fin.get('total_ar', 0):,.0f}
            - AUDIT RISKS: {audit.get('findings', [])}
            
            Give me 3 DIRECT ORDERS to save the company.
            Format: "<b>ORDER:</b> [Action] - [Specific Deadline/Metric]"
            Do not summarize. Just orders."""
            
            res = self.llm.invoke(prompt)
            return AgentResult({"orders": res.content}, datetime.now(), True)
        except:
            return AgentResult({"orders": "<b>ORDER:</b> Manual Strategy Required (AI Offline)."}, datetime.now(), False)

# ================================================================
# 6. REPORT GENERATOR
# ================================================================
class CEOReportGenerator:
    def __init__(self):
        self.viz = VisualizerEngine()
        self.styles = getSampleStyleSheet()
        
    def generate(self, data, filename):
        doc = SimpleDocTemplate(filename, pagesize=letter, topMargin=40, bottomMargin=40)
        story = []
        
        audit = data['audit']
        fin = data['finance']
        strat = data['strategy']
        
        # Styles
        title_style = ParagraphStyle('T', parent=self.styles['Heading1'], fontSize=24, textColor=colors.HexColor('#2c3e50'), alignment=1)
        h2_style = ParagraphStyle('H2', parent=self.styles['Heading2'], fontSize=16, textColor=colors.HexColor('#c0392b'), spaceBefore=15, borderBottomWidth=1, borderColor=colors.grey)
        
        # 1. HEADER
        story.append(Paragraph("ODIN COMMAND CENTER", title_style))
        story.append(Paragraph(f"Period: {data['period']} | Generated: {datetime.now().strftime('%Y-%m-%d %H:%M')}", self.styles['Normal']))
        story.append(Spacer(1, 20))
        
        # 2. SURVIVAL METRICS (The "Am I Dying?" Section)
        story.append(Paragraph("1. SURVIVAL METRICS", h2_style))
        story.append(Spacer(1, 10))
        
        # Profit vs Cash + Runway Chart
        img_paradox = RLImage(self.viz.plot_profit_paradox(fin['profit'], fin['net_cash_flow']), width=3.5*inch, height=2.2*inch)
        img_runway = RLImage(self.viz.plot_cash_runway(fin['net_cash_flow'], fin['burn_rate']), width=3.5*inch, height=2.2*inch)
        
        # Simple table layout for charts
        data_table = [[img_paradox, img_runway]]
        t = Table(data_table)
        story.append(t)
        story.append(Spacer(1, 10))
        
        # 3. LIQUIDITY & DEBTORS (The "Who Owes Me?" Section)
        story.append(Paragraph("2. ACCOUNTS RECEIVABLE DRILL-DOWN", h2_style))
        story.append(Spacer(1, 10))
        
        img_debtors = RLImage(self.viz.plot_debtor_breakdown(fin['debtors']), width=5*inch, height=3*inch)
        story.append(img_debtors)
        story.append(Paragraph(f"<b>Total Uncollected:</b> ${fin['total_ar']:,.2f}", self.styles['BodyText']))
        story.append(Spacer(1, 10))
        
        # 4. OPERATIONAL INTEGRITY (The "Audit" Section)
        story.append(Paragraph("3. OPERATIONAL RISKS (AUDIT)", h2_style))
        story.append(Spacer(1, 10))
        
        img_inv = RLImage(self.viz.plot_inventory_risk(audit['good_stock'], audit['ghost_stock']), width=4*inch, height=3*inch)
        story.append(img_inv)
        
        if audit['findings']:
            for find in audit['findings']:
                story.append(Paragraph(f"• <font color='red'>{find}</font>", self.styles['Normal']))
        
        story.append(PageBreak())
        
        # 5. STRATEGIC ORDERS
        story.append(Paragraph("4. CEO DIRECTIVES", h2_style))
        story.append(Spacer(1, 10))
        
        orders_text = strat.get('orders', '')
        # Basic parsing
        for line in orders_text.split('\n'):
            if line.strip():
                story.append(Paragraph(line, self.styles['BodyText']))
                story.append(Spacer(1, 8))

        doc.build(story)
        return filename

# ================================================================
# 7. MAIN EXECUTION
# ================================================================
if __name__ == "__main__":
    print("🚀 ODIN v8.0 Command Center - Initializing...")
    
    # Init
    auditor = AuditorAgent()
    financialist = FinancialistAgent()
    strategist = StrategistAgent()
    reporter = CEOReportGenerator()
    
    # Dates (Last 90 Days)
    end_date = datetime.now().strftime('%Y-%m-%d')
    start_date = (datetime.now() - timedelta(days=90)).strftime('%Y-%m-%d')
    
    # Run Workflow
    print("🔍 Auditor scanning integrity...")
    res_audit = auditor.execute(start_date, end_date)
    
    print("💸 Financialist analyzing runway & debtors...")
    res_fin = financialist.execute(start_date, end_date)
    
    print("🧠 Strategist formulating orders...")
    res_strat = strategist.execute(res_audit.data, res_fin.data)
    
    # Compile
    full_data = {
        "audit": res_audit.data,
        "finance": res_fin.data,
        "strategy": res_strat.data,
        "period": f"{start_date} to {end_date}"
    }
    
    # Report
    print("📊 Rendering Command Center PDF...")
    fpath = f"{Config.REPORT_DIR}/ODIN_Command_Center.pdf"
    reporter.generate(full_data, fpath)
    
    print(f"✅ REPORT READY: {fpath}")

✅ Connected to Odoo
🚀 ODIN v8.0 Command Center - Initializing...
🔍 Auditor scanning integrity...
💸 Financialist analyzing runway & debtors...
🧠 Strategist formulating orders...
📊 Rendering Command Center PDF...
✅ REPORT READY: ./odin_reports/ODIN_Command_Center.pdf
